# 02 — Model Training & Evaluation

**Task:** predict district-level paddy yield (kg/ha) for each season.

**Main model:** **Explainable Boosting Machine (EBM)**, which was not covered in the lectures. It is compared with:
- a naive baseline (the district's recent average)
- Ridge regression, the linear model from the lectures
- tuned **Random Forest** and **XGBoost**, the strongest ensemble models from the lectures

**Evaluation:**
- The data is split **by time**, so the model is always tested on future seasons it has never seen.
- Hyperparameters are tuned on a separate validation period.
- The test set is used **once**, at the end.

In [ ]:
from pathlib import Path
import time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid, ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from interpret.glassbox import ExplainableBoostingRegressor

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG, MODELS = ROOT / "reports" / "figures", ROOT / "models"
FIG.mkdir(parents=True, exist_ok=True); MODELS.mkdir(exist_ok=True)

COLORS = {"Naive baseline": "#b5b4ae", "Ridge": "#e87ba4", "Random Forest": "#1baf7a",
          "XGBoost": "#eda100", "EBM": "#2a78d6"}
SEASON_COLORS = {"Maha": "#2a78d6", "Yala": "#eb6834"}
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "font.size": 10, "lines.linewidth": 2, "figure.dpi": 110,
})
def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight"); plt.show()

df = pd.read_csv(ROOT / "data_processed" / "paddy_dataset.csv")
df.shape

## 1. Features

The features come from the EDA in notebook 01. Columns that are only known **after harvest** are excluded to avoid leakage:
- harvested area
- harvest ratio
- the separate yields for each irrigation type

`sown_total` is log-transformed because district paddy areas range from tens of hectares to more than 100,000 ha.

In [ ]:
CATEGORICAL = ["district", "season"]
NUMERIC = ["year", "log_sown_total", "sown_share_major", "sown_share_minor", "sown_share_rainfed",
           "rain_total_mm", "rain_min_month_mm", "rain_max_month_mm", "rain_cv", "rain_pre_season_mm",
           "temp_mean_c", "temp_max_c", "temp_min_c", "humidity_pct", "solar_mj_m2_day", "wind_m_s"]
FEATURES = CATEGORICAL + NUMERIC
TARGET = "yield_total"

df["log_sown_total"] = np.log1p(df["sown_total"])
print(len(FEATURES), "features:", FEATURES)

## 2. Time-based train / validation / test split

| Set | Harvest years | Purpose |
|---|---|---|
| Train | 1981–2017 | fit the models |
| Validation | 2018–2020 | choose hyperparameters |
| Test | 2021–2025 | final, one-time evaluation |

A random split would put the same district and year in both train and test. That would **leak** the time trend and inflate the scores.

**Why `fertilizer_shock` is not a feature:** the flag is 1 only in 2022, which falls in the **test** period. The model never sees a
fertilizer shock during training, so it cannot learn its effect. We keep the flag only to **analyse** errors in the test
set. This is a realistic limitation: no model can predict an unprecedented policy shock from weather alone.

In [ ]:
train = df[df.year <= 2017].copy()
val   = df[(df.year >= 2018) & (df.year <= 2020)].copy()
test  = df[df.year >= 2021].copy()
for name, d in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s}: {len(d):5d} rows | years {d.year.min()}-{d.year.max()} | mean yield {d[TARGET].mean():,.0f}")
print("fertilizer_shock rows in train/val/test:", train.fertilizer_shock.sum(), val.fertilizer_shock.sum(), test.fertilizer_shock.sum())

X_tr, y_tr = train[FEATURES], train[TARGET]
X_va, y_va = val[FEATURES], val[TARGET]
X_te, y_te = test[FEATURES], test[TARGET]

## 3. Evaluation helper

In [ ]:
def metrics(y_true, y_pred):
    return {"RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE": mean_absolute_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred),
            "MAPE_%": np.mean(np.abs((y_true - y_pred) / y_true)) * 100}

def tune(make_model, candidates, name):
    """Fit every candidate on train, score on validation, return best params + table."""
    rows = []
    t0 = time.time()
    for params in candidates:
        m = make_model(**params).fit(X_tr, y_tr)
        rows.append({**params, **metrics(y_va, m.predict(X_va))})
    res = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    print(f"{name}: {len(rows)} candidates tuned in {time.time()-t0:.0f}s | best validation RMSE {res.RMSE[0]:,.0f}")
    return clean_params(res.loc[0, list(candidates[0])].to_dict()), res

INT_PARAMS = {"n_estimators", "max_depth", "min_samples_leaf", "min_child_weight", "interactions", "max_bins"}
def clean_params(p):
    """DataFrame storage turns ints into floats and None into NaN - undo that."""
    out = {}
    for k, v in p.items():
        if v is None or (isinstance(v, float) and np.isnan(v)):
            out[k] = None
        elif k in INT_PARAMS:
            out[k] = int(v)
        else:
            out[k] = float(v)
    return out

## 4. Baselines

- **Naive baseline:** predict each district-season's average yield over its last 5 training seasons.
  This is what a planner would use without any model. A useful model must beat it.
- **Ridge regression:** the regularised linear model from the lectures. Missing values are filled with the median,
  district and season are one-hot encoded, and the numeric features are scaled.

In [ ]:
def naive_predict(history, target_rows, n=5):
    recent = (history.sort_values("year").groupby(["district", "season"])[TARGET]
              .apply(lambda s: s.tail(n).mean()))
    overall = history.groupby("season")[TARGET].mean()
    return np.array([recent.get((d, s), overall[s]) for d, s in zip(target_rows.district, target_rows.season)])

print("Naive baseline (validation):", {k: round(v, 3) for k, v in metrics(y_va, naive_predict(train, val)).items()})

def preprocessor(scale):
    num_steps = [("impute", SimpleImputer(strategy="median"))] + ([("scale", StandardScaler())] if scale else [])
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
        ("num", Pipeline(num_steps), NUMERIC)])

def make_ridge(alpha):
    return Pipeline([("prep", preprocessor(scale=True)), ("model", Ridge(alpha=alpha))])

ridge_best, ridge_res = tune(make_ridge, list(ParameterGrid({"alpha": [0.1, 1, 3, 10, 30, 100]})), "Ridge")
ridge_res.head()

## 5. Random Forest (tuned)

- **Random search:** 25 combinations, each scored on the validation years.
- **Search space:** number of trees, tree depth, minimum leaf size and features per split.
- We do **not** use the default parameters, as the lecturer requested.

In [ ]:
def make_rf(**p):
    return Pipeline([("prep", preprocessor(scale=False)),
                     ("model", RandomForestRegressor(random_state=SEED, n_jobs=-1, **p))])

rf_space = {"n_estimators": [300, 500, 800], "max_depth": [None, 10, 15, 20, 30],
            "min_samples_leaf": [1, 2, 4, 8], "max_features": [0.3, 0.5, 0.7, 1.0]}
rf_best, rf_res = tune(make_rf, list(ParameterSampler(rf_space, n_iter=25, random_state=SEED)), "Random Forest")
print("Best RF params:", rf_best)
rf_res.head()

## 6. XGBoost (tuned)

- **Random search:** 40 combinations.
- **Search space:** learning rate, depth, number of trees, row and column subsampling, minimum child weight and L2 regularisation.

In [ ]:
def make_xgb(**p):
    return Pipeline([("prep", preprocessor(scale=False)),
                     ("model", XGBRegressor(random_state=SEED, n_jobs=-1, tree_method="hist", **p))])

xgb_space = {"n_estimators": [300, 600, 1000, 1500], "learning_rate": [0.01, 0.03, 0.05, 0.1],
             "max_depth": [3, 4, 5, 6, 8], "subsample": [0.6, 0.8, 1.0], "colsample_bytree": [0.5, 0.7, 1.0],
             "min_child_weight": [1, 3, 5, 10], "reg_lambda": [0.5, 1, 5, 10]}
xgb_best, xgb_res = tune(make_xgb, list(ParameterSampler(xgb_space, n_iter=40, random_state=SEED)), "XGBoost")
print("Best XGB params:", xgb_best)
xgb_res.head()

## 7. Explainable Boosting Machine (tuned), the new algorithm

**How EBM works:** EBM is a *Generalised Additive Model with interactions* (GA²M). It learns:

$$\hat{y} = \beta_0 + \sum_i f_i(x_i) + \sum_{i,j} f_{ij}(x_i, x_j)$$

- Each $f_i$ is a **shape function** for one feature, learned with gradient boosting. Each boosting round trains tiny trees on **one feature at a time**, cycling through the features.
- A small number of pairwise **interactions** $f_{ij}$ are detected and added automatically.
- The final model is a **sum of one-feature curves**, so every prediction can be broken down **exactly** into feature contributions.

This makes EBM a *glass-box* model, unlike Random Forest and XGBoost, which are black boxes.

**Search space:** learning rate, number of interactions, minimum samples per leaf and maximum bins.

`district` and `season` are passed to EBM as native categorical features, with no one-hot encoding needed.

In [ ]:
def make_ebm(**p):
    return ExplainableBoostingRegressor(random_state=SEED, n_jobs=-1,
                                        feature_types=["nominal", "nominal"] + ["continuous"] * len(NUMERIC), **p)

ebm_space = {"learning_rate": [0.01, 0.03], "interactions": [0, 10, 20],
             "min_samples_leaf": [2, 10], "max_bins": [128, 256]}
ebm_best, ebm_res = tune(make_ebm, list(ParameterGrid(ebm_space)), "EBM")
print("Best EBM params:", ebm_best)
ebm_res.head()

## 8. Validation comparison

All models are compared on the **validation** years (2018–2020) using their best hyperparameters.

In [ ]:
val_scores = pd.DataFrame({
    "Naive baseline": metrics(y_va, naive_predict(train, val)),
    "Ridge": ridge_res.iloc[0][["RMSE", "MAE", "R2", "MAPE_%"]].to_dict(),
    "Random Forest": rf_res.iloc[0][["RMSE", "MAE", "R2", "MAPE_%"]].to_dict(),
    "XGBoost": xgb_res.iloc[0][["RMSE", "MAE", "R2", "MAPE_%"]].to_dict(),
    "EBM": ebm_res.iloc[0][["RMSE", "MAE", "R2", "MAPE_%"]].to_dict(),
}).T.astype(float)
val_scores.round(3)

## 9. Final models: refit on train + validation, evaluate once on test

Each model is refit with its best hyperparameters on **1981–2020** and then evaluated **once** on **2021–2025**.

We report the test scores three ways:
- on the full test period
- on the **normal seasons only**, excluding the 2022 fertilizer-shock seasons the model could not have learned
- on the **shock seasons only**

In [ ]:
trval = pd.concat([train, val])
X_trval, y_trval = trval[FEATURES], trval[TARGET]

final = {
    "Ridge": make_ridge(**ridge_best),
    "Random Forest": make_rf(**rf_best),
    "XGBoost": make_xgb(**xgb_best),
    "EBM": make_ebm(**ebm_best),
}
preds = {"Naive baseline": naive_predict(trval, test)}
fit_times = {}
for name, model in final.items():
    t0 = time.time(); model.fit(X_trval, y_trval); fit_times[name] = time.time() - t0
    preds[name] = model.predict(X_te)

normal = test.fertilizer_shock.values == 0
rows = []
for name, p in preds.items():
    rows.append({"model": name, "subset": "All test (2021-25)", **metrics(y_te.values, p)})
    rows.append({"model": name, "subset": "Normal seasons", **metrics(y_te.values[normal], p[normal])})
    rows.append({"model": name, "subset": "Fertilizer shock (2022)", **metrics(y_te.values[~normal], p[~normal])})
test_scores = pd.DataFrame(rows)
summary = test_scores.pivot(index="model", columns="subset", values=["RMSE", "MAE", "R2"]).round(2)
summary = summary.reindex(["Naive baseline", "Ridge", "Random Forest", "XGBoost", "EBM"])
print("Fit time (s):", {k: round(v, 1) for k, v in fit_times.items()})
summary

In [ ]:
order = ["Naive baseline", "Ridge", "Random Forest", "XGBoost", "EBM"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["RMSE", "MAE", "R2"]):
    d = test_scores[test_scores.subset == "Normal seasons"].set_index("model").loc[order, metric]
    bars = ax.bar(order, d.values, color=[COLORS[m] for m in order], width=0.6)
    for b, v in zip(bars, d.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.2f}" if metric == "R2" else f"{v:,.0f}",
                ha="center", va="bottom", fontsize=9, color=INK)
    ax.set_title(metric + (" (higher is better)" if metric == "R2" else " (kg/ha, lower is better)"), fontsize=11)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="x", visible=False)
fig.suptitle("Test performance, normal seasons 2021-2025 (excl. fertilizer shock)", fontweight="bold", color=INK)
fig.tight_layout()
save(fig, "model_01_test_metrics")

## 10. Predicted vs actual (test set, EBM)

In [ ]:
best_name = "EBM"
p = preds[best_name]
fig, ax = plt.subplots(figsize=(6.2, 6))
for s, c in SEASON_COLORS.items():
    m = (test.season.values == s) & normal
    ax.scatter(y_te.values[m], p[m], s=22, alpha=0.6, color=c, edgecolor="none", label=f"{s}")
ax.scatter(y_te.values[~normal], p[~normal], s=34, facecolor="none", edgecolor="#e34948", lw=1.3,
           label="Fertilizer-shock seasons (2022)")
lims = [1000, 7000]
ax.plot(lims, lims, color=INK2, lw=1, ls="--"); ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual yield (kg/ha)"); ax.set_ylabel("Predicted yield (kg/ha)")
r = test_scores.query("model == @best_name and subset == 'Normal seasons'").iloc[0]
ax.set_title(f"EBM: predicted vs actual (test)\nnormal seasons RMSE {r.RMSE:,.0f}, R² {r.R2:.2f}")
ax.legend(frameon=False, loc="upper left")
save(fig, "model_02_ebm_pred_vs_actual")

## 11. National yield, actual vs predicted, per test season

In [ ]:
t = test.assign(**{f"pred_{k}": v for k, v in preds.items()})
w = t.harvested_total
nat = t.groupby("season_label").apply(lambda g: pd.Series(
    {"Actual": np.average(g[TARGET], weights=g.harvested_total),
     **{k: np.average(g[f"pred_{k}"], weights=g.harvested_total) for k in ["EBM", "XGBoost", "Random Forest"]}}))
key = {lab: (int(lab[:4]) + (0.5 if "Maha" in lab else 0)) for lab in nat.index}
nat = nat.loc[sorted(nat.index, key=lambda l: key[l])]

fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(nat))
ax.plot(x, nat["Actual"], color=INK, marker="o", ms=7, label="Actual")
for k in ["EBM", "XGBoost", "Random Forest"]:
    ax.plot(x, nat[k], color=COLORS[k], marker="o", ms=5, lw=1.6, label=k)
ax.set_xticks(x, nat.index, rotation=30, ha="right")
ax.set_ylabel("Area-weighted yield (kg/ha)")
ax.set_title("National yield per test season: actual vs predicted")
ax.legend(frameon=False, ncol=4, loc="lower right")
save(fig, "model_03_national_test_seasons")
nat.round(0)

## 12. Errors by district (EBM, normal test seasons)

In [ ]:
err = test[normal].assign(error=(preds["EBM"] - y_te.values)[normal])
by_d = err.groupby("district").error.agg(["mean", lambda e: np.sqrt(np.mean(e**2))]).rename(columns={"mean": "bias", "<lambda_0>": "RMSE"})
by_d = by_d.sort_values("RMSE")
fig, ax = plt.subplots(figsize=(7, 7.5))
ax.barh(by_d.index, by_d.RMSE, color="#2a78d6", height=0.6)
ax.set_xlabel("RMSE (kg/ha)"); ax.set_title("EBM test error by district (normal seasons)")
ax.grid(axis="y", visible=False)
save(fig, "model_04_ebm_error_by_district")
by_d.round(0).tail(6)

## 13. Save models and outputs for notebook 03 and the app

In [ ]:
for name, model in final.items():
    joblib.dump(model, MODELS / f"{name.lower().replace(' ', '_')}.joblib")
meta = {"features": FEATURES, "categorical": CATEGORICAL, "numeric": NUMERIC, "target": TARGET,
        "train_years": [int(trval.year.min()), int(trval.year.max())], "test_years": [2021, 2025],
        "best_params": {"Ridge": ridge_best, "Random Forest": rf_best, "XGBoost": xgb_best, "EBM": ebm_best},
        "districts": sorted(df.district.unique().tolist())}
(MODELS / "model_meta.json").write_text(json.dumps(meta, indent=2, default=str))
test_scores.round(4).to_csv(ROOT / "reports" / "test_scores.csv", index=False)
val_scores.round(4).to_csv(ROOT / "reports" / "validation_scores.csv")
t.drop(columns=[c for c in t.columns if c.startswith("pred_") and c != "pred_EBM"]).to_csv(
    ROOT / "data_processed" / "test_predictions.csv", index=False)
print(sorted(p.name for p in MODELS.iterdir()))

## 14. Results and discussion

_(filled in after running)_